# Mental Health Text Classifier — Prototype
**Leveraging LLM Technology for Early Detection of Mental Health Signals**

Student ID: 20211453  
Chapters covered: 1 · 2 · 3 · 4 · 5 · 6 · 7

---
| Section | Chapter | Technique |
|---|---|---|
| 0 | Setup | Install & clone |
| 1 | Ch.1 | Zero-shot LLM baseline |
| 2 | Ch.2 | Anchor embedding classifier |
| 3 | Ch.3 | Hidden state extraction |
| 4 | Ch.4 | Core classifier (train + eval) |
| 5 | Ch.5 | Unsupervised clustering |
| 6 | Ch.6 | Prompt engineering pipeline |
| 7 | Ch.7 | Memory & empathy system |
| 8 | — | Full pipeline demo |
| 9 | — | Results & visualisation |

## Section 0 — Setup

In [ ]:
# Install dependencies
!pip install -q torch transformers accelerate sentence-transformers scikit-learn \
    datasets umap-learn hdbscan bertopic joblib \
    llama-cpp-python huggingface_hub \
    langchain langchain-community \
    matplotlib seaborn pandas

In [ ]:
import os, sys

REPO_URL = "https://github.com/CJiu01/mental-health-classifier.git"
if not os.path.exists("mental-health-classifier"):
    !git clone {REPO_URL}

os.chdir("mental-health-classifier")
sys.path.insert(0, ".")
print("Working directory:", os.getcwd())

## Section 1 — Ch.1: Zero-Shot LLM Baseline

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed:", torch.cuda.memory_allocated() / 1e9, "GB in use")

In [ ]:
from modules.ch1_zero_shot import ZeroShotClassifier

ch1 = ZeroShotClassifier()

samples = [
    "I feel so happy and grateful today!",
    "I am really worried and can't stop thinking about the worst.",
    "I want to disappear and stop being a burden to everyone.",
]
print("=== Ch.1 Zero-Shot Results ===")
for s in samples:
    out = ch1.predict(s)
    print(f"  [{out['risk_level']:8}]  '{s[:55]}'")
    print(f"            {out['reasoning']}\n")

## Section 2 — Ch.2: Anchor Embedding Classifier

In [ ]:
from modules.ch2_anchor import AnchorClassifier

ch2 = AnchorClassifier()

print("=== Ch.2 Anchor Cosine Results ===")
for s in samples:
    out = ch2.predict(s)
    print(f"  [{out['risk_level']:8}]  score={out['risk_score']:.3f}  '{s[:50]}'")
    print(f"            Anchor scores: {out['emotion_scores']}\n")

## Section 3 — Ch.3: Hidden State Feature Extraction

In [ ]:
from modules.ch3_hidden_state import extract_hidden_state

print("=== Ch.3 Hidden State Vectors ===")
for s in samples:
    vec = extract_hidden_state(s)
    print(f"  shape={tuple(vec.shape)}  norm={vec.norm():.3f}  '{s[:50]}'")

## Section 4 — Ch.4: Core Classifier (Train + Evaluate)

In [ ]:
from data.loader import load_emotion_dataset, print_stats

data = load_emotion_dataset()
print_stats(data)

In [ ]:
from modules.ch4_classifier import MentalHealthClassifier

clf = MentalHealthClassifier()
clf.train(data["train"]["texts"], data["train"]["labels"])
clf.save()

In [ ]:
print("=== Ch.4 Test Set Evaluation ===")
report = clf.evaluate(data["test"]["texts"], data["test"]["labels"])

In [ ]:
print("=== Ch.4 Sample Predictions ===")
for s in samples:
    out = clf.predict(s)
    print(f"  [{out['risk_level']:8}]  score={out['risk_score']:.3f}  "
          f"emotion={out['primary_emotion']:8}  '{s[:45]}'")
    print(f"            scores: {out['emotion_scores']}\n")

## Section 5 — Ch.5: Unsupervised Emotional Clustering

In [ ]:
from modules.ch5_clustering import EmotionClusterer
import numpy as np

# Use first 5,000 train texts for speed
texts_5k = data["train"]["texts"][:5000]
print("Encoding 5,000 texts with ch4 shared encoder...")
embeddings_5k = clf.encoder.encode(texts_5k, show_progress_bar=True, batch_size=64)

clusterer = EmotionClusterer()
clusterer.fit(texts_5k, embeddings_5k)
print("\nTopic info (top 10 clusters):")
print(clusterer.get_topic_info().head(10))

In [ ]:
print("=== Ch.5 Sample Cluster Analysis ===")
risk_samples = [
    "I feel so hopeless and don't see any reason to continue.",
    "I had such a wonderful day with my friends!",
]
for s in risk_samples:
    result = clusterer.transform(s)
    print(f"  '{s[:55]}'")
    print(f"  Cluster: {result['cluster_id']}  "
          f"Keywords: {result['cluster_keywords']}  "
          f"IsRisk: {result['is_risk_cluster']}\n")

In [ ]:
# Visualise clusters in 2D
import matplotlib.pyplot as plt
from umap import UMAP
import pandas as pd

reduced_2d = UMAP(n_components=2, min_dist=0.0, metric='cosine',
                  random_state=42).fit_transform(embeddings_5k)
labels_5k = data["train"]["labels"][:5000]
label_names = data["label_names"]

df_vis = pd.DataFrame(reduced_2d, columns=["x","y"])
df_vis["emotion"] = [label_names[l] for l in labels_5k]

fig, ax = plt.subplots(figsize=(9,6))
colors = {"sadness":"#3498db","joy":"#f1c40f","love":"#e91e63",
          "anger":"#e74c3c","fear":"#9b59b6","surprise":"#2ecc71"}
for emotion, grp in df_vis.groupby("emotion"):
    ax.scatter(grp.x, grp.y, s=2, alpha=0.5,
               c=colors[emotion], label=emotion)
ax.legend(markerscale=5, loc="upper right")
ax.set_title("UMAP 2D — Emotion Clusters (Ch.5)")
ax.axis("off")
plt.tight_layout()
plt.savefig("umap_clusters.png", dpi=150)
plt.show()

## Section 6 — Ch.6: Prompt Engineering Pipeline

In [ ]:
from modules.ch6_prompt import PromptClassifier
import json

ch6 = PromptClassifier()

print("=== Ch.6 Prompt Engineering Results ===")
for s in samples:
    out = ch6.predict(s)
    print(f"  [{out['risk_level']:8}]  '{s[:50]}'")
    print(f"  Emotion : {out.get('primary_emotion')}")
    print(f"  Reason  : {out.get('reasoning', '')[:120]}")
    print(f"  Advice  : {out.get('recommendation', '')[:120]}\n")

## Section 7 — Ch.7: Memory & Empathy System

In [ ]:
from modules.ch7_memory import EmotionalMemorySystem

# Scenario A: Deteriorating trend
mem_a = EmotionalMemorySystem(session_id="scenario_a")
scenario_a = [
    ("I had a great day today!",               "Positive", 0.10, "joy"),
    ("Feeling pretty good, got things done.",  "Positive", 0.15, "joy"),
    ("Normal day, nothing special.",           "Neutral",  0.25, "surprise"),
    ("Feeling a bit low today.",               "Neutral",  0.35, "sadness"),
    ("Really tired and anxious lately.",       "Negative", 0.55, "fear"),
    ("Can't sleep, everything feels hopeless.","Negative", 0.65, "sadness"),
    ("I just want to disappear.",              "Crisis",   0.82, "sadness"),
]
for text, risk, score, emotion in scenario_a:
    mem_a.add_entry(text, risk, score, emotion)

print("Scenario A — Deteriorating")
print("  Trend    :", mem_a.get_trend())
print("  Direction:", mem_a.get_trend_direction())

In [ ]:
# Scenario B: Improving trend
mem_b = EmotionalMemorySystem(session_id="scenario_b")
scenario_b = [
    ("I want to disappear.",             "Crisis",   0.85, "sadness"),
    ("Can't stop crying.",               "Crisis",   0.80, "sadness"),
    ("Feeling a bit less hopeless.",     "Negative", 0.55, "sadness"),
    ("Had a small win today.",           "Neutral",  0.30, "surprise"),
    ("Things are looking up.",           "Positive", 0.12, "joy"),
    ("Feeling hopeful again.",           "Positive", 0.08, "joy"),
    ("Had a wonderful day!",             "Positive", 0.06, "joy"),
]
for text, risk, score, emotion in scenario_b:
    mem_b.add_entry(text, risk, score, emotion)

print("Scenario B — Improving")
print("  Trend    :", mem_b.get_trend())
print("  Direction:", mem_b.get_trend_direction())

In [ ]:
# Visualise 7-day trend (Figure 4)
import matplotlib.pyplot as plt
from config import RISK_WEIGHT

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
risk_colors = {"Positive":"#2ecc71","Neutral":"#95a5a6",
               "Negative":"#e67e22","Crisis":"#e74c3c"}

for ax, (mem, title) in zip(axes, [
    (mem_a, "Scenario A: Deteriorating"),
    (mem_b, "Scenario B: Improving"),
]):
    trend  = mem.get_trend()
    scores = [RISK_WEIGHT[r] for r in trend]
    colors = [risk_colors[r] for r in trend]
    ax.plot(range(1, len(scores)+1), scores, 'o-', color='#2c3e50', lw=2)
    for i, (s, c) in enumerate(zip(scores, colors)):
        ax.scatter(i+1, s, color=c, s=120, zorder=5)
    ax.set_xticks(range(1, len(scores)+1))
    ax.set_xticklabels([f"D{i}" for i in range(1, len(scores)+1)])
    ax.set_yticks([0,1,2,3])
    ax.set_yticklabels(["Positive","Neutral","Negative","Crisis"])
    ax.set_title(f"{title}\nDirection: {mem.get_trend_direction()}")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("trend_simulation.png", dpi=150)
plt.show()

## Section 8 — Full Pipeline Demo

In [ ]:
from pipeline import MentalHealthPipeline

pipe = MentalHealthPipeline(mode="quick")

print("=== Full Pipeline — Quick Mode ===")
for s in samples:
    out = pipe.run(s)
    print(f"  [{out['risk_level']:8}]  score={out['risk_score']:.3f}  "
          f"trend={out['trend']}  dir={out['trend_direction']}")
    print(f"            '{s[:55]}'\n")

In [ ]:
# Full mode (ch4 + ch6 + ch7) — requires GPU
pipe_full = MentalHealthPipeline(mode="full")

print("=== Full Pipeline — Full Mode ===")
diary_entries = [
    "Today was exhausting. I barely got out of bed.",
    "Had a rough week. Can't stop worrying about everything.",
    "I don't see the point anymore. Everything feels heavy.",
]
for entry in diary_entries:
    out = pipe_full.run(entry)
    print(f"\n  Input   : '{entry}'")
    print(f"  Risk    : {out['risk_level']} (score={out['risk_score']:.3f})")
    print(f"  Reason  : {out.get('reasoning','')[:100]}")
    print(f"  Advice  : {out.get('recommendation','')[:100]}")
    print(f"  Empathy : {out.get('empathy_response','')[:100]}")
    print(f"  Trend   : {out['trend']} → {out['trend_direction']}")

## Section 9 — Results & Visualisation

In [ ]:
from evaluate import (
    exp1_core_evaluation,
    exp2_model_comparison,
    exp3_risk_distribution,
)

# Exp 1: Core classifier
report = exp1_core_evaluation(clf, data["test"]["texts"], data["test"]["labels"])

In [ ]:
# Exp 2: Model comparison (ch2 vs ch4; skip ch1 for speed)
comparison_df = exp2_model_comparison(
    data["test"]["texts"],
    data["test"]["labels"],
    ch4_clf=clf,
    ch2_clf=ch2,
    ch1_clf=None,   # set ch1_clf=ch1 to include (slow)
)
comparison_df

In [ ]:
# Exp 3: Risk distribution
exp3_risk_distribution(data["test"]["texts"], data["test"]["labels"], clf)